# Solar System Demo

A live preview of what you'll be able to build by the end of this course —
eight planets orbiting the sun, Saturn's rings, the Moon circling Earth,
dramatic lighting, and Bach playing in the background.

Press **▶** to run the cell, then sit back and watch.

<a target="_blank" href="https://codetto.app?github=simonguest/codercub/blob/main/cmu/00/notebooks/space_demo.ipynb">
  <img src="https://img.shields.io/badge/Open_in-Codetto-blue" alt="Open In Codetto"/>
</a>

In [ ]:
from codetto import scene3d, audio
import math

scene = scene3d.Scene()
scene.set_sky(scene3d.Sky.DEEP_SPACE)
scene.ambient.set_brightness(35)

# Sun light (no sphere — the planets are the stars of the show)
sun_light = scene.add_light(0, 0, 0)
sun_light.set_color('#ffeeaa')
sun_light.set_brightness(5)

# Planets: (material, orbital_radius, speed, diameter, initial_angle)
planet_data = [
    (scene3d.Material.Planets.Mercury,  9,  2.0,  1.5,  0.0),
    (scene3d.Material.Planets.Venus,   14,  1.3,  2.5,  1.2),
    (scene3d.Material.Planets.Earth,   20,  0.9,  2.5,  2.5),
    (scene3d.Material.Planets.Mars,    27,  0.65, 2.0,  0.8),
    (scene3d.Material.Planets.Jupiter, 44,  0.3,  9.0,  3.8),
    (scene3d.Material.Planets.Uranus,  68,  0.15, 5.5,  1.5),
    (scene3d.Material.Planets.Neptune, 80,  0.10, 5.5,  4.2),
]

planets = []
for mat, r, speed, dia, angle0 in planet_data:
    p = scene3d.Shapes.Sphere(diameter=dia, segments=24)
    p.set_material(mat)
    p.set_position(r * math.cos(angle0), 0, r * math.sin(angle0))
    p.set_glossiness(0)
    scene.add(p)
    planets.append((p, r, speed, angle0))

# Saturn with rings
saturn_group = scene3d.Group()
saturn_body = scene3d.Shapes.Sphere(diameter=8.0, segments=24)
saturn_body.set_material(scene3d.Material.Planets.Saturn)
saturn_ring = scene3d.Shapes.Cylinder(diameter=17.0, height=0.3, tessellation=32)
#saturn_ring.set_color('#c8a464')
saturn_ring.set_material(scene3d.Material.Gravel.LightGray)
saturn_ring.set_tiling(5)
saturn_ring.set_rotation(x=72)
saturn_group.add(saturn_body)
saturn_group.add(saturn_ring)
scene.add(saturn_group)

# Moon orbiting Earth
moon = scene3d.Shapes.Sphere(diameter=0.8, segments=16)
moon.set_material(scene3d.Material.Planets.Moon)
scene.add(moon)

# Title HUD
ctx = scene.get_context('2d')
ctx.fill_style = 'rgba(255,255,255,0.85)'
ctx.font = 'bold 22px sans-serif'
ctx.fill_text('Solar System  —  Python with 3D', 14, 38)

t = 0.0
SATURN_R     = 56
SATURN_SPEED = 0.2
SATURN_A0    = 2.8

# Camera visits each planet in sequence, smoothly panning between them
VISIT     = [2,   3,   4,   -1,  5,   6,   1,   0  ]  # -1 = Saturn
ORBIT_R   = [10,  8,   20,  24,  12,  12,  8,   6  ]  # camera orbit radius per planet
ORBIT_H   = [6,   5,   12,  14,  7,   7,   5,   4  ]  # camera height per planet
PHASE_DUR = 10                                          # seconds per planet

cam = {'lx': 15.0, 'ly': 0.0, 'lz': 0.0, 'r': 30.0, 'h': 12.0, 'ang': 0.0}

@scene.on_frame
def animate(dt):
    global t
    t += dt

    for p, r, speed, angle0 in planets:
        angle = angle0 + t * speed * 0.22
        p.set_position(r * math.cos(angle), 0, r * math.sin(angle))

    sa = SATURN_A0 + t * SATURN_SPEED * 0.22
    saturn_group.set_position(
        SATURN_R * math.cos(sa), 0,
        SATURN_R * math.sin(sa)
    )

    earth = planets[2][0]
    ex, ey, ez = earth.get_position()
    ma = t * 3.5
    moon.set_position(ex + 5 * math.cos(ma), 0, ez + 5 * math.sin(ma))

    # Determine which planet to focus on and desired orbit params
    phase_idx  = int(t / PHASE_DUR) % len(VISIT)
    planet_idx = VISIT[phase_idx]
    desired_r  = ORBIT_R[phase_idx]
    desired_h  = ORBIT_H[phase_idx]

    if planet_idx == -1:
        tx, ty, tz = saturn_group.get_position()
    else:
        tx, ty, tz = planets[planet_idx][0].get_position()

    # Smoothly pan look-at and adjust orbit radius toward the new target
    look_blend  = min(1.0, dt * 2.5)
    orbit_blend = min(1.0, dt * 1.2)

    cam['lx'] += (tx - cam['lx']) * look_blend
    cam['ly'] += (ty - cam['ly']) * look_blend
    cam['lz'] += (tz - cam['lz']) * look_blend
    cam['r']  += (desired_r - cam['r'])  * orbit_blend
    cam['h']  += (desired_h - cam['h'])  * orbit_blend
    cam['ang'] += dt * 0.22

    scene.camera.set_position(
        cam['lx'] + cam['r'] * math.cos(cam['ang']),
        cam['h'],
        cam['lz'] + cam['r'] * math.sin(cam['ang'])
    ).look_at(cam['lx'], cam['ly'], cam['lz'])

await audio.play_async('/sample_files/music_bach.wav')
scene.run()